In [ ]:
import numpy as np
import joblib
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
df = pd.read_csv(
    "../data/processed/featured_uhi_v2.csv"
)

df.head()

In [ ]:
# 1. Spatial blocks: cut the city into a 10x10 grid
n_bins = 10
df["lat_bin"] = pd.cut(df["Latitude"], bins=n_bins, labels=False)
df["lon_bin"] = pd.cut(df["Longitude"], bins=n_bins, labels=False)
df["spatial_block"] = df["lat_bin"].astype(str) + "_" + df["lon_bin"].astype(str)

# 2. Physics-only features (no Lat/Long, no engineered duplicates)
FEATURES = ["NDVI", "NDBI", "Elevation", "Population"]
X = df[FEATURES]
y = df["LST"]
groups = df["spatial_block"]

# 3. Spatial cross-validation with out-of-fold predictions
gkf = GroupKFold(n_splits=5)
oof_pred = np.zeros(len(y))

for tr, te in gkf.split(X, y, groups):
    m = RandomForestRegressor(n_estimators=300, random_state=42)
    m.fit(X.iloc[tr], y.iloc[tr])
    oof_pred[te] = m.predict(X.iloc[te])

mae  = mean_absolute_error(y, oof_pred)
rmse = mean_squared_error(y, oof_pred) ** 0.5
r2   = r2_score(y, oof_pred)

print("Random Forest V3 - Spatial CV (honest generalisation)")
print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R2   : {r2:.3f}")

# 4. Train final model on ALL data, save for later notebooks
final_model = RandomForestRegressor(n_estimators=300, random_state=42)
final_model.fit(X, y)
joblib.dump(final_model, "../outputs/lst_model_v3.pkl")
print("\nSaved -> outputs/lst_model_v3.pkl")

In [ ]:
comparison = pd.DataFrame({"Actual": y, "Predicted": oof_pred})
comparison.head(10)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.scatter(y, oof_pred, s=8, alpha=0.3)
plt.xlabel("Actual LST")
plt.ylabel("Predicted LST (out-of-fold)")
plt.title("Random Forest V3 - Spatial CV")
plt.show()